# Barreiras e Fila Distribuídas com ZooKeeper

Este notebook apresenta a implementação de primitivas de sincronização (barreiras e fila) utilizando o ZooKeeper. Cada seção contém o código acompanhado de explicações. Os testes para cada componente foram separados em células individuais, evitando que os logs se misturem e facilitando a análise dos resultados.

## Importação das Bibliotecas Necessárias

Nesta célula são importadas as bibliotecas essenciais para o funcionamento do código, incluindo módulos padrão do Python e a biblioteca `kazoo` para conexão com o ZooKeeper. Certifique-se de que o ZooKeeper esteja rodando em `127.0.0.1:2181` ou ajuste o endereço conforme necessário.

In [119]:
import threading
import time
import socket
import uuid
import random
from kazoo.client import KazooClient

# Observação: ajuste o endereço do ZooKeeper se necessário.

## 1. SyncPrimitive

A classe `SyncPrimitive` é a base para todas as primitivas de sincronização. Ela cria e mantém uma instância compartilhada do cliente ZooKeeper e utiliza um mutex (via `threading.Condition`) para coordenar a sincronização entre threads.

In [120]:
class SyncPrimitive:
    _zk: KazooClient = None  # Instância compartilhada do cliente ZooKeeper
    _mutex = threading.Condition()  # Mutex para sincronização entre threads

    def __init__(self, address):
        if SyncPrimitive._zk is None:
            SyncPrimitive._zk = KazooClient(address)  # Conecta ao ZooKeeper
            SyncPrimitive._zk.start()  # Inicia a conexão
        self.address = address

    @property
    def zk(self):
        return SyncPrimitive._zk  # Retorna a instância do ZooKeeper

## 2. BaseBarrier

A classe `BaseBarrier` herda de `SyncPrimitive` e contém funcionalidades comuns para todas as barreiras, como garantir a existência do nó da barreira e definir um _watch callback_ que notifica as threads quando há alterações.

In [121]:
class BaseBarrier(SyncPrimitive):
    def __init__(self, address, path):
        super().__init__(address)  # Inicializa a conexão com o ZooKeeper
        self.path = path  # Caminho do nó que representa a barreira
        self._ensure_path()  # Garante que o nó da barreira exista

    def _ensure_path(self):
        if not self.zk.exists(self.path):
            self.zk.ensure_path(self.path)  # Cria o nó se não existir

    def _watch_callback(self, event):
        # Callback chamado quando há mudanças no nó observado
        with SyncPrimitive._mutex:
            SyncPrimitive._mutex.notify_all()  # Notifica todas as threads em espera

## 3. SingleBarrier

A `SingleBarrier` é uma barreira simples em que as threads aguardam até que o nó da barreira seja removido. O método `enter` entra em loop verificando a existência do nó e o `leave` o remove, notificando as threads para prosseguir.

In [122]:
class SingleBarrier(BaseBarrier):
    def enter(self):
        while True:
            with SyncPrimitive._mutex:
                if not self.zk.exists(self.path, watch=self._watch_callback):
                    return  # Sai se o nó não existir
                SyncPrimitive._mutex.wait()  # Aguarda notificação

    def leave(self):
        with SyncPrimitive._mutex:
            if self.zk.exists(self.path):
                self.zk.delete(self.path, recursive=True)  # Remove o nó
            SyncPrimitive._mutex.notify_all()  # Notifica as threads

## 4. DoubleBarrier

A `DoubleBarrier` sincroniza a entrada e a saída dos participantes. Cada participante cria um nó efêmero com um ID único. Em `enter`, o participante aguarda até que o número de nós filhos seja igual ou superior ao esperado; em `leave`, remove seu nó e aguarda até que todos saiam.

In [123]:
class DoubleBarrier(BaseBarrier):
    def __init__(self, address, root, size):
        super().__init__(address, root)  # Inicializa com o nó raiz
        self.size = size  # Número esperado de participantes
        self.node_path = None  # Caminho do nó efêmero deste participante

    def enter(self):
        hostname = socket.gethostname()  # Nome do host
        unique_id = f"{hostname}-{uuid.uuid4()}"  # ID único
        self.node_path = f"{self.path}/{unique_id}"
        self.zk.create(self.node_path, ephemeral=True)  # Cria nó efêmero

        while True:
            with SyncPrimitive._mutex:
                children = self.zk.get_children(self.path, watch=self._watch_callback)  
                if len(children) < self.size:
                    SyncPrimitive._mutex.wait()
                else:
                    return True

    def leave(self):
        if self.node_path:
            with SyncPrimitive._mutex:
                if self.zk.exists(self.node_path):
                    self.zk.delete(self.node_path)
                SyncPrimitive._mutex.notify_all()

        while True:
            with SyncPrimitive._mutex:
                children = self.zk.get_children(self.path, watch=self._watch_callback)
                if len(children) > 0:
                    SyncPrimitive._mutex.wait()
                else:
                    return True

## 5. Queue

A classe `Queue` implementa uma fila distribuída usando o ZooKeeper. Cada elemento é adicionado como um nó sequencial sob um nó raiz. O método `consume` ordena os nós e remove o elemento mais antigo, garantindo a ordem de chegada.

In [124]:
class Queue(SyncPrimitive):
    def __init__(self, address, root):
        super().__init__(address)  # Conexão com o ZooKeeper
        self.root = root  # Nó raiz da fila
        self._ensure_root()  

    def _ensure_root(self):
        self.zk.ensure_path(self.root)  # Cria o nó raiz, se necessário

    def produce(self, value):
        # Adiciona um elemento criando um nó sequencial
        self.zk.create(f"{self.root}/element-", str(value).encode(), sequence=True)

    def consume(self):
        while True:
            children = self.zk.get_children(self.root)  
            if not children:
                time.sleep(1)  # Aguarda se a fila estiver vazia
                continue
            sorted_children = sorted(children)
            first_child = sorted_children[0]
            data, _ = self.zk.get(f"{self.root}/{first_child}")
            self.zk.delete(f"{self.root}/{first_child}")
            return int(data.decode())

## 6. RestrictedBarrier

A classe `RestrictedBarrier` limita o número de clientes que podem entrar simultaneamente na barreira. Se o número de participantes já atingir o limite (`max_clients`), uma exceção é lançada. Cada cliente cria um nó efêmero sequencial em um subcaminho específico.

In [125]:
class RestrictedBarrier(BaseBarrier):
    def __init__(self, address, path, max_clients):
        super().__init__(address, path)
        self.max_clients = max_clients
        self.client_path = f"{self.path}/clients"
        self._ensure_path()
        
    def enter(self):
        if not self.zk.exists(self.client_path):
            self.zk.create(self.client_path, makepath=True)
        children = self.zk.get_children(self.client_path)
        if len(children) < self.max_clients:
            node_path = f"{self.client_path}/client-"
            self.zk.create(node_path, ephemeral=True, sequence=True)
        else:
            raise Exception("Número máximo de clientes na barreira atingido.")

    def leave(self):
        children = self.zk.get_children(self.client_path)
        for child in children:
            full_path = f"{self.client_path}/{child}"
            if self.zk.exists(full_path):
                self.zk.delete(full_path)
                break

## 7. ReusableBarrier

A `ReusableBarrier` permite que a mesma barreira seja utilizada em múltiplos ciclos. Cada participante cria um nó efêmero único e, ao atingirem o número esperado, todos podem prosseguir. Após a saída, a barreira pode ser reiniciada com o método `reset`.

In [126]:
class ReusableBarrier(BaseBarrier):
    def __init__(self, address, root, size):
        super().__init__(address, root)
        self.size = size
        self._nodes = {}  # Dicionário para armazenar, por thread, o caminho do nó efêmero

    def enter(self):
        """
        Cada thread cria seu próprio nó efêmero e aguarda até que o número de nós
        no caminho da barreira seja igual ou superior ao tamanho esperado.
        """
        hostname = socket.gethostname()
        unique_id = f"{hostname}-{uuid.uuid4()}"
        node_path = f"{self.path}/{unique_id}"
        # Armazena o nó criado para a thread atual
        self._nodes[threading.get_ident()] = node_path
        self.zk.create(node_path, ephemeral=True)

        while True:
            with SyncPrimitive._mutex:
                children = self.zk.get_children(self.path, watch=self._watch_callback)
                if len(children) >= self.size:
                    return True
                SyncPrimitive._mutex.wait()

    def leave(self):
        """
        A thread remove seu próprio nó efêmero se ele existir e notifica as demais.
        """
        node_path = self._nodes.pop(threading.get_ident(), None)
        if node_path and self.zk.exists(node_path):
            try:
                self.zk.delete(node_path)
            except Exception:
                # Se o nó já foi removido, ignoramos o erro
                pass
        with SyncPrimitive._mutex:
            SyncPrimitive._mutex.notify_all()

    def reset(self):
        """
        Reinicia completamente a barreira, removendo recursivamente o nó raiz e recriando-o.
        """
        self.zk.delete(self.path, recursive=True)
        self._ensure_path()


## 8. Definição das Funções de Teste

A seguir, definimos as funções de teste para cada uma das primitivas implementadas. Note que os testes foram adaptados para execução interativa no notebook (sem uso de argumentos de linha de comando).

In [127]:
# Teste para SingleBarrier
def test_single_barrier():
    address = "127.0.0.1:2181"
    barrier_path = "/single_barrier_test"
    barrier = SingleBarrier(address, barrier_path)

    def worker(idx):
        print(f"[Thread-{idx}] Aguardando liberação da barreira...")
        barrier.enter()
        print(f"[Thread-{idx}] Passou da barreira!")

    threads = []
    for i in range(3):
        t = threading.Thread(target=worker, args=(i,))
        t.start()
        threads.append(t)

    time.sleep(2)
    print("[Main] Removendo a barreira...")
    barrier.leave()

    for t in threads:
        t.join()
    
    print("[Main] Teste SingleBarrier concluído.")

# Teste para DoubleBarrier
def test_double_barrier():
    address = "127.0.0.1:2181"
    root = "/double_barrier_test"
    size = 3

    def worker(idx):
        barrier = DoubleBarrier(address, root, size)
        print(f"[Thread-{idx}] -> enter()")
        barrier.enter()
        print(f"[Thread-{idx}] Todos chegaram. Executando tarefa...")
        time.sleep(1 + idx * 0.2)
        print(f"[Thread-{idx}] -> leave()")
        barrier.leave()
        print(f"[Thread-{idx}] Saiu da barreira!")

    threads = []
    for i in range(size):
        t = threading.Thread(target=worker, args=(i,))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()
    
    print("[Main] Teste DoubleBarrier concluído.")

# Teste para Queue (Produtor e Consumidor)
def test_queue():
    address = "127.0.0.1:2181"
    root = "/queue_test"
    queue = Queue(address, root)

    print("[PRODUTOR] Produzindo elementos...")
    for i in range(5):
        queue.produce(i)
        print(f"Elemento {i} adicionado.")

    print("[CONSUMIDOR] Consumindo elementos...")
    for i in range(5):
        item = queue.consume()
        print(f"Consumido: {item}")

# Teste para RestrictedBarrier
def test_restricted_barrier():
    address = "127.0.0.1:2181"
    barrier_path = "/restricted_barrier_test"
    max_size = 3

    barrier = RestrictedBarrier(address, barrier_path, max_size)

    def worker(idx):
        print(f"[Thread-{idx}] Tentando entrar na barreira...")
        barrier.enter()
        print(f"[Thread-{idx}] Passou da barreira e executando tarefa...")
        time.sleep(1 + idx * 0.2)
        print(f"[Thread-{idx}] Saindo da barreira...")
        barrier.leave()
        print(f"[Thread-{idx}] Saiu da barreira com sucesso!")

    threads = []
    for i in range(max_size):
        t = threading.Thread(target=worker, args=(i,))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()
    
    print("[Main] Teste RestrictedBarrier concluído.")

# Teste para ReusableBarrier
def test_reusable_barrier(zk_address="127.0.0.1:2181", root_path="/reusable_barrier_test", group_size=3):
    barrier = ReusableBarrier(zk_address, root_path, group_size)

    def test_cycle(cycle_num):
        print(f"\n=== Ciclo {cycle_num} Iniciado ===")

        def worker():
            print(f"[Thread {threading.get_ident()}] Tentando entrar")
            barrier.enter()
            print(f"[Thread {threading.get_ident()}] Dentro da barreira")
            time.sleep(1)
            barrier.leave()
            print(f"[Thread {threading.get_ident()}] Saiu")

        threads = []
        for _ in range(barrier.size):
            t = threading.Thread(target=worker)
            threads.append(t)
            t.start()

        for t in threads:
            t.join()

        print(f"=== Ciclo {cycle_num} Concluído ===\n")

    for cycle in range(1, 4):
        test_cycle(cycle)

    barrier.reset()
    print("✅ Teste ReusableBarrier concluído!")

## 9. Execução dos Testes em Células Separadas

Cada teste abaixo foi colocado em uma célula individual para facilitar a análise dos logs. Execute cada célula separadamente para visualizar os resultados de cada teste.

### 9.1. Teste: SingleBarrier

In [128]:
test_single_barrier()

[Thread-0] Aguardando liberação da barreira...
[Thread-1] Aguardando liberação da barreira...
[Thread-2] Aguardando liberação da barreira...
[Main] Removendo a barreira...
[Thread-2] Passou da barreira!
[Thread-0] Passou da barreira!
[Thread-1] Passou da barreira!
[Main] Teste SingleBarrier concluído.


### 9.2. Teste: DoubleBarrier

In [129]:
test_double_barrier()

[Thread-0] -> enter()
[Thread-1] -> enter()
[Thread-2] -> enter()
[Thread-1] Todos chegaram. Executando tarefa...
[Thread-0] Todos chegaram. Executando tarefa...
[Thread-2] Todos chegaram. Executando tarefa...
[Thread-0] -> leave()
[Thread-1] -> leave()
[Thread-2] -> leave()
[Thread-2] Saiu da barreira!
[Thread-0] Saiu da barreira!
[Thread-1] Saiu da barreira!
[Main] Teste DoubleBarrier concluído.


### 9.3. Teste: Queue (Produtor e Consumidor)

In [130]:
test_queue()

[PRODUTOR] Produzindo elementos...
Elemento 0 adicionado.
Elemento 1 adicionado.
Elemento 2 adicionado.
Elemento 3 adicionado.
Elemento 4 adicionado.
[CONSUMIDOR] Consumindo elementos...
Consumido: 0
Consumido: 1
Consumido: 2
Consumido: 3
Consumido: 4


### 9.4. Teste: RestrictedBarrier

In [131]:
test_restricted_barrier()

[Thread-0] Tentando entrar na barreira...[Thread-1] Tentando entrar na barreira...

[Thread-2] Tentando entrar na barreira...
[Thread-0] Passou da barreira e executando tarefa...
[Thread-1] Passou da barreira e executando tarefa...
[Thread-2] Passou da barreira e executando tarefa...
[Thread-0] Saindo da barreira...
[Thread-0] Saiu da barreira com sucesso!
[Thread-1] Saindo da barreira...
[Thread-1] Saiu da barreira com sucesso!
[Thread-2] Saindo da barreira...
[Thread-2] Saiu da barreira com sucesso!
[Main] Teste RestrictedBarrier concluído.


### 9.5. Teste: ReusableBarrier

In [132]:
test_reusable_barrier()


=== Ciclo 1 Iniciado ===
[Thread 13120450560] Tentando entrar
[Thread 13137276928] Tentando entrar
[Thread 13154103296] Tentando entrar
[Thread 13120450560] Dentro da barreira
[Thread 13137276928] Dentro da barreira
[Thread 13154103296] Dentro da barreira
[Thread 13137276928] Saiu
[Thread 13120450560] Saiu
[Thread 13154103296] Saiu
=== Ciclo 1 Concluído ===


=== Ciclo 2 Iniciado ===
[Thread 13120450560] Tentando entrar
[Thread 13137276928] Tentando entrar
[Thread 13154103296] Tentando entrar
[Thread 13120450560] Dentro da barreira
[Thread 13137276928] Dentro da barreira
[Thread 13154103296] Dentro da barreira
[Thread 13120450560] Saiu
[Thread 13137276928] Saiu
[Thread 13154103296] Saiu
=== Ciclo 2 Concluído ===


=== Ciclo 3 Iniciado ===
[Thread 13120450560] Tentando entrar
[Thread 13137276928] Tentando entrar
[Thread 13154103296] Tentando entrar
[Thread 13120450560] Dentro da barreira
[Thread 13137276928] Dentro da barreira
[Thread 13154103296] Dentro da barreira
[Thread 13120450560